# 06 種の文の方向で「右−左」の物差しを作れるか

**問い**: 日本語 BERT の埋め込みで、右寄りの文と左寄りの文の差(「右−左」の向き)を物差しにすれば、国会の発言から党の立場を測れるか。

**方法**: 右寄り・左寄りの「種の文」を自分で書き、BERT(`tohoku-nlp/bert-base-japanese-v3`、mean pooling)で埋め込む。種の文の差を物差しにして、次の順に確かめる。

1. 争点をまたいで共通の「右−左」の向きがあるか(セル2)
2. 否定の言い方(「〜すべきではありません」)に引っ張られないか(セル3・4)
3. 実際の発言(安全保障の委員会、6,172件)に当てたときの党の並び(セル5・6)
4. 種の文を入れ替えても党の並びが変わらないか、何を拾っているか(セル7)

**結論(先に)**: 物差しは立場よりも**話し方(批判的か協力的か)や話題**を強く拾っており、種の文を半分入れ替えただけで党の並びが大きく変わる(相関 0.66)。この方法では立場を安定して取り出せない → 会議録の BERT 埋め込みは党の位置の推定に使わない(マニフェストの政策ごとの判定を主にする理由の1つ)。

## 準備:BERT と道具の関数

- `embed`: 文を768次元のベクトルにする。会議録を埋め込んだ Colab(04)と同じ方法(CLS・SEP を含めた mean pooling)にそろえる
- `cos`: 2つのベクトルの向きがどれくらい同じか(1 に近いほど同じ)
- `unit`: ベクトルの長さを1にそろえる(物差しの向きだけを使うため)

In [1]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

MODEL = "tohoku-nlp/bert-base-japanese-v3"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL).eval()

@torch.no_grad()
def embed(texts):
    """Colab と同じ方法(CLS・SEP を含めた mean pooling)で文を埋め込む"""
    enc = tok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    h = model(**enc).last_hidden_state
    m = enc["attention_mask"].unsqueeze(-1).float()
    return ((h * m).sum(1) / m.sum(1)).numpy()

def cos(a, b):
    """コサイン類似度(向きがどれくらい同じか。1に近いほど同じ向き)"""
    return (a * b).sum(-1) / (np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1))

def unit(v):
    """長さを1にそろえる"""
    return v / np.linalg.norm(v)

print(embed(["テストの文です。"]).shape)   # (1, 768)

c:\Users\taiyo\anaconda3\envs\survey_on_party_systems\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4897.73it/s]
[transformers] BertModel LOAD REPORT from: tohoku-nlp/bert-base-japanese-v3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias

(1, 768)


## 1. 争点をまたいで共通の「右−左」の向きがあるか

5つの争点(憲法・防衛・原発・夫婦別姓・経済)で、右寄りと左寄りの文を1組ずつ作る。

- (A) 自然な文の組: ふつうに書いた右と左の文
- (B) 否定だけが違う組: 「〜すべきです」と「〜すべきではありません」だけが違う文。夫婦別姓と経済だけは、右寄りの側が否定の形

見るもの: ① 右と左の文がどれくらい似ているか ② 争点ごとの「右−左」の向きがそろっているか(そろっていれば共通の物差しがある)③ 4争点で作った物差しで、残り1争点の右と左を正しく分けられるか

In [2]:
# (争点, 右寄りの文, 左寄りの文)
natural = [
    ("憲法",     "憲法を改正して、自衛隊の存在を明記すべきだと考えます。",
                 "憲法九条を守り、平和主義を堅持すべきだと考えます。"),
    ("防衛",     "防衛力を抜本的に強化し、防衛費を増額すべきです。",
                 "防衛費の増額ではなく、外交による平和の構築を優先すべきです。"),
    ("原発",     "原子力発電所を再稼働し、エネルギーの安定供給を確保すべきです。",
                 "原子力発電所は再稼働せず、再生可能エネルギーに転換すべきです。"),
    ("夫婦別姓", "家族の一体感を守るため、夫婦同姓の制度を維持すべきです。",
                 "個人の尊重のため、選択的夫婦別姓制度を導入すべきです。"),
    ("経済",     "規制緩和と成長戦略によって、経済の活力を高めるべきです。",
                 "格差を是正するため、富裕層への課税を強化すべきです。"),
]
negation = [
    ("憲法",     "憲法を改正すべきです。",                     "憲法を改正すべきではありません。"),
    ("防衛",     "防衛費を増額すべきです。",                   "防衛費を増額すべきではありません。"),
    ("原発",     "原子力発電所を再稼働すべきです。",           "原子力発電所を再稼働すべきではありません。"),
    ("夫婦別姓", "選択的夫婦別姓を導入すべきではありません。", "選択的夫婦別姓を導入すべきです。"),
    ("経済",     "富裕層への課税を強化すべきではありません。", "富裕層への課税を強化すべきです。"),
]

def check(pairs, name):
    topics = [t for t, _, _ in pairs]
    Rc = embed([r for _, r, _ in pairs])
    Lc = embed([l for _, _, l in pairs])
    print(f"\n========== {name} ==========")
    same = cos(Rc, Lc)
    other = np.mean([cos(Rc[i], Rc[j]) for i in range(5) for j in range(5) if i != j])
    print("① 同じ争点の右と左の類似度:", dict(zip(topics, same.round(3))))
    print("   (参考)違う争点の文どうしの類似度の平均:", round(other, 3))
    D = Rc - Lc
    Dn = D / np.linalg.norm(D, axis=1, keepdims=True)
    S = Dn @ Dn.T
    off = S[~np.eye(5, dtype=bool)]
    print("② 「右−左」の向きの一致度(争点どうしの平均):", round(off.mean(), 3))
    print(pd.DataFrame(S, index=topics, columns=topics).round(2))
    print("③ 1つ抜きのテスト")
    for i in range(5):
        ax = D[[j for j in range(5) if j != i]].mean(0)
        ok = Rc[i] @ ax > Lc[i] @ ax
        print(f"   {topics[i]}: {'○' if ok else '×'}")

check(natural, "(A) 自然な文の組")
check(negation, "(B) 否定だけが違う文の組")


========== (A) 自然な文の組 ==========
① 同じ争点の右と左の類似度: {'憲法': np.float32(0.897), '防衛': np.float32(0.919), '原発': np.float32(0.965), '夫婦別姓': np.float32(0.947), '経済': np.float32(0.924)}
   (参考)違う争点の文どうしの類似度の平均: 0.846
② 「右−左」の向きの一致度(争点どうしの平均): -0.026
        憲法    防衛    原発  夫婦別姓    経済
憲法    1.00  0.39 -0.16 -0.34 -0.13
防衛    0.39  1.00  0.09 -0.15 -0.24
原発   -0.16  0.09  1.00  0.23 -0.03
夫婦別姓 -0.34 -0.15  0.23  1.00  0.09
経済   -0.13 -0.24 -0.03  0.09  1.00
③ 1つ抜きのテスト
   憲法: ×
   防衛: ○
   原発: ○
   夫婦別姓: ×
   経済: ×

========== (B) 否定だけが違う文の組 ==========
① 同じ争点の右と左の類似度: {'憲法': np.float32(0.935), '防衛': np.float32(0.954), '原発': np.float32(0.96), '夫婦別姓': np.float32(0.957), '経済': np.float32(0.962)}
   (参考)違う争点の文どうしの類似度の平均: 0.846
② 「右−左」の向きの一致度(争点どうしの平均): -0.151
        憲法    防衛    原発  夫婦別姓    経済
憲法    1.00  0.86  0.82 -0.77 -0.78
防衛    0.86  1.00  0.84 -0.80 -0.86
原発    0.82  0.84  1.00 -0.78 -0.78
夫婦別姓 -0.77 -0.80 -0.78  1.00  0.74
経済   -0.78 -0.86 -0.78  0.74  1.00
③ 1つ抜きのテスト
   憲法: ○
   防衛: ○
   原発:

**結果**

- (A) 争点ごとの「右−左」の向きは、平均すると **−0.03**(ほぼ無関係)。共通の「右−左」の向きは見つからない。1つ抜きのテストも 5問中2問しか当たらない
- (B) 否定だけが違う組では、向きが **±0.8 前後**で強くそろう。ただし、右が肯定の3争点どうしは +0.8、右が否定の2争点とは −0.8 → BERT は**右か左かではなく「否定の形かどうか」を強い1つの向きとして捉えている**
- → 種の文の差は、立場よりも言い方(否定の有無)を拾いやすい

## 2. 否定の形をそろえると、どうなるか(安全保障だけ)

安全保障に争点をしぼり、10組の文を作る。1〜5組は右が肯定・左が否定、6〜10組は右が否定・左が肯定にして、**否定の形の偏りを打ち消す**。

- ① 1つ抜きのテスト: k組目と k+5組目を一緒に抜いて物差しを作り、抜いた2組を分けられるか
- ② 物差しと「否定の方角」(否定の文 − 肯定の文)がどれくらい同じ向きか(0 に近いほど否定に引っ張られていない)

In [3]:
# 1〜5組:右が肯定・左が否定 / 6〜10組:右が否定・左が肯定
pairs_sec = [
    ("防衛費をさらに増額すべきです。",               "防衛費をさらに増額すべきではありません。"),
    ("憲法を改正して自衛隊を明記すべきです。",       "憲法を改正して自衛隊を明記すべきではありません。"),
    ("反撃能力の保有を進めるべきです。",             "反撃能力の保有を進めるべきではありません。"),
    ("日米同盟をいっそう強化すべきです。",           "日米同盟をいっそう強化すべきではありません。"),
    ("防衛装備品の輸出を拡大すべきです。",           "防衛装備品の輸出を拡大すべきではありません。"),
    ("防衛費の増額を見送るべきではありません。",     "防衛費の増額を見送るべきです。"),
    ("憲法改正の議論を先送りすべきではありません。", "憲法改正の議論を先送りすべきです。"),
    ("反撃能力の保有をためらうべきではありません。", "反撃能力の保有をためらうべきです。"),
    ("日米同盟への協力を弱めるべきではありません。", "日米同盟への協力を弱めるべきです。"),
    ("防衛装備品の輸出を制限すべきではありません。", "防衛装備品の輸出を制限すべきです。"),
]
R = embed([r for r, _ in pairs_sec])
L = embed([l for _, l in pairs_sec])

print("① 1つ抜きのテスト(k組目と k+5組目を一緒に抜く)")
hits = 0
for k in range(5):
    test = [k, k + 5]
    train = [j for j in range(10) if j not in test]
    ax = R[train].mean(0) - L[train].mean(0)
    for t in test:
        ok = bool(R[t] @ ax > L[t] @ ax)
        hits += int(ok)
        print(f"   {pairs_sec[t][0][:18]:<20} {'○' if ok else '×'}")
print(f"   正解: {hits} / 10")

neg_dir = np.vstack([L[:5], R[5:]]).mean(0) - np.vstack([R[:5], L[5:]]).mean(0)   # 否定の文 − 肯定の文
print("\n② 物差しと「否定の方角」の類似度:", round(float(cos(R.mean(0) - L.mean(0), neg_dir)), 3))

① 1つ抜きのテスト(k組目と k+5組目を一緒に抜く)
   防衛費をさらに増額すべきです。      ○
   防衛費の増額を見送るべきではありませ   ×
   憲法を改正して自衛隊を明記すべきです   ○
   憲法改正の議論を先送りすべきではあり   ×
   反撃能力の保有を進めるべきです。     ○
   反撃能力の保有をためらうべきではあり   ○
   日米同盟をいっそう強化すべきです。    ○
   日米同盟への協力を弱めるべきではあり   ○
   防衛装備品の輸出を拡大すべきです。    ○
   防衛装備品の輸出を制限すべきではあり   ×
   正解: 7 / 10

② 物差しと「否定の方角」の類似度: -0.221


**結果**: 正解 **7 / 10**。外れた3つはすべて「右が否定の形」の文(6〜10組目)。否定の方角との類似度は −0.22 → 否定の形をそろえても、まだ否定に引っ張られている

## 3. 言葉づかいも実際の議論に近づけた種の文

国会の議論で実際に使われそうな言い回しで、右10文・左10文を作る(それぞれ前半5文が肯定、後半5文が否定の形)。

- ① 1つ抜きのテスト(1文ずつ抜いて、抜いた文を分けられるか)
- ② 物差しと「否定の方角」の類似度

In [4]:
# 右10・左10。それぞれ前半5文が肯定、後半5文が否定の形
right_frame = [
    "我が国を取り巻く安全保障環境は戦後最も厳しく、抑止力を抜本的に強化しなければなりません。",
    "防衛力の抜本的強化と防衛費の増額を着実に進めるべきです。",
    "反撃能力を保有し、日米同盟の抑止力と対処力を高めるべきです。",
    "自衛隊の存在を憲法に明記し、国防の基盤を確かなものにすべきです。",
    "周辺国の軍事的な台頭に対し、防衛体制の強化を急ぐべきです。",
    "厳しさを増す安全保障環境から目を背けてはなりません。",
    "防衛力の強化をこれ以上先延ばしにしてはなりません。",
    "抑止力の向上なくして、国民の命と平和な暮らしは守れません。",
    "日米同盟の信頼性を損なうような対応をとるべきではありません。",
    "憲法改正の議論から逃げるべきではありません。",
]
left_frame = [
    "憲法九条に基づく平和外交こそ、我が国が進むべき道です。",
    "専守防衛の原則を守り、対話による緊張緩和を図るべきです。",
    "防衛費の大幅な増額よりも、暮らしと社会保障に予算を回すべきです。",
    "敵基地攻撃能力の保有は憲法違反であり、撤回すべきです。",
    "軍拡競争ではなく、近隣諸国との信頼醸成を進めるべきです。",
    "憲法九条をないがしろにする大軍拡は許されません。",
    "専守防衛の原則を投げ捨てるべきではありません。",
    "軍事力に頼るだけでは、平和は実現できません。",
    "国民の暮らしを犠牲にした防衛費の増額は認められません。",
    "集団的自衛権の行使を際限なく広げるべきではありません。",
]
Rf = embed(right_frame)
Lf = embed(left_frame)

print("① 1つ抜きのテスト")
hits = 0
for i in range(10):
    train = [j for j in range(10) if j != i]
    ax = Rf[train].mean(0) - Lf[train].mean(0)
    ok_r = bool(Rf[i] @ ax > Lf[train].mean(0) @ ax)
    ok_l = bool(Lf[i] @ ax < Rf[train].mean(0) @ ax)
    hits += int(ok_r) + int(ok_l)          # NumPy の True どうしの + は「または」になるので、int にしてから足す
    print(f"   {i+1:>2}組目  右:{'○' if ok_r else '×'}  左:{'○' if ok_l else '×'}")
print(f"   正解: {hits} / 20")

axis_frame = Rf.mean(0) - Lf.mean(0)
print("\n② 物差しと「否定の方角」の類似度:", round(float(cos(axis_frame, neg_dir)), 3))

① 1つ抜きのテスト
    1組目  右:○  左:○
    2組目  右:○  左:○
    3組目  右:○  左:○
    4組目  右:○  左:○
    5組目  右:○  左:○
    6組目  右:○  左:○
    7組目  右:○  左:○
    8組目  右:○  左:○
    9組目  右:○  左:○
   10組目  右:×  左:○
   正解: 19 / 20

② 物差しと「否定の方角」の類似度: -0.04


**結果**: 正解 **19 / 20**、否定の方角との類似度は **−0.04**(否定にはほぼ引っ張られていない)。種の文どうしでは、うまく右と左を分けられる物差しができた。→ 次に、この物差し(`axis_frame`、以下「言葉づかい」)を実際の発言に当てる

## 4. 実際の発言に当てる

会議録(02 で前処理・04 で埋め込み済み)から、安全保障の委員会(安全保障委員会・外交防衛委員会・外務委員会・憲法審査会)の100字以上の発言を選ぶ。各発言を2つの物差しに当て、全体で平均0・標準偏差1にそろえて、党ごとに平均する。

- 否定そろえ: セル3の物差し
- 言葉づかい: セル4の物差し

In [5]:
EMB = Path("../data/embeddings/emb")
X = np.load(EMB / "embeddings.npy").astype(np.float32)
ids = pd.read_csv(EMB / "speech_ids.csv")
q = pd.read_parquet("../data/processed/speeches.parquet",
                    columns=["speechID", "date", "party", "nameOfHouse", "nameOfMeeting", "text"])
keep_ids = ids["speechID"].isin(q["speechID"]).to_numpy()
X = X[keep_ids]
ids = ids[keep_ids].reset_index(drop=True)
df = ids.merge(q, on="speechID", how="left", validate="one_to_one")
df["n_char"] = df["text"].str.len()

SEC = r"安全保障委員会|外交防衛委員会|外務委員会|憲法審査会"
sel = ((df["n_char"] >= 100) & df["nameOfMeeting"].str.contains(SEC, na=False)).to_numpy()
print("対象の発言:", sel.sum(), "件")

axes = {
    "否定そろえ": unit(R.mean(0) - L.mean(0)),
    "言葉づかい": unit(axis_frame),
}
res = df.loc[sel, ["party"]].copy()
for name, ax in axes.items():
    s = X[sel] @ ax
    res[name] = (s - s.mean()) / s.std()

tab = res.groupby("party").agg(件数=("party", "size"),
                               否定そろえ=("否定そろえ", "mean"),
                               言葉づかい=("言葉づかい", "mean"))
print(tab[tab["件数"] >= 30].sort_values("言葉づかい").round(2))

対象の発言: 6172 件
         件数  否定そろえ  言葉づかい
party                    
社民      158  -0.95  -1.34
共産      561  -1.01  -0.81
その他     200  -0.96  -0.56
れいわ     117  -0.45  -0.53
立憲     1741   0.04  -0.10
維新      694   0.02  -0.08
国民      863   0.26   0.23
中道      387   0.22   0.34
自民      609   0.37   0.38
参政      257   0.12   0.51
みらい     117   0.50   0.59
公明      465   0.55   0.67


**結果**: 対象 6,172件。「言葉づかい」の物差しで並べると、社民・共産・れいわが左、自民・公明・参政が右で、見た目はそれらしい並びになる。ただし、これだけでは本当に立場を測っているかはわからない(次のセルで確かめる)

## 5. 会議ごとの違いを除き、連立再編の前後で比べる

会議(院 + 委員会名)ごとに話し方や話題が違うので、会議ごとの平均を引いた値(会議調整)も出す。公明の連立離脱(2025-10-10)の前と後に分けて、党ごとに平均する。

In [6]:
res["会議"] = df.loc[sel, "nameOfHouse"] + df.loc[sel, "nameOfMeeting"]
res["date"] = df.loc[sel, "date"]
res["言葉_会議調整"] = res["言葉づかい"] - res.groupby("会議")["言葉づかい"].transform("mean")
res["after"] = res["date"] >= "2025-10-10"

t = (res.groupby(["party", "after"])
        .agg(件数=("party", "size"),
             言葉=("言葉づかい", "mean"),
             会議調整=("言葉_会議調整", "mean"))
        .unstack())
print(t.round(2))

           件数           言葉        会議調整      
after   False  True  False True  False True 
party                                       
その他     199.0    1.0 -0.56 -0.29 -0.41  0.20
みらい       NaN  117.0   NaN  0.59   NaN  0.46
れいわ      90.0   27.0 -0.36 -1.12 -0.28 -0.90
中道        NaN  387.0   NaN  0.34   NaN  0.17
保守        NaN    3.0   NaN -0.84   NaN -0.91
公明      267.0  198.0  0.88  0.39  0.92  0.46
共産      296.0  265.0 -0.70 -0.93 -0.66 -0.87
参政        NaN  257.0   NaN  0.51   NaN  0.48
国民      341.0  522.0 -0.08  0.44 -0.12  0.40
社民        3.0  155.0 -2.18 -1.32 -1.66 -1.22
立憲     1257.0  484.0 -0.11 -0.08 -0.17 -0.03
維新      462.0  232.0 -0.08 -0.08 -0.09 -0.03
自民      354.0  255.0  0.18  0.64  0.28  0.72


**結果**: 公明は離脱の前 0.88(会議調整 0.92)→ 後 0.39(0.46)に下がり、自民は 0.18 → 0.64 に上がる。立場の変化というより、**与党側か野党側か(政府への質問が協力的か批判的か)で動いている可能性**がある。社民・保守など件数がごく少ない組み合わせは当てにならない

## 6. 物差しは安定しているか、何を拾っているか

- (1) セル4の種の文を半分ずつに分けて物差しを2本作り(半分A・半分B)、党の並びが同じになるかを見る。同じ立場を測っているなら、相関は 1 に近いはず
- (2) 維新の発言のうち、「左寄り」「右寄り」と判定された発言を3件ずつ読む

In [7]:
# (1) 種の文を半分に分けて物差しを2本作り、党の並びを比べる
half_a = [0, 2, 4, 6, 8]
half_b = [1, 3, 5, 7, 9]
axA = unit(Rf[half_a].mean(0) - Lf[half_a].mean(0))
axB = unit(Rf[half_b].mean(0) - Lf[half_b].mean(0))
sA, sB = X[sel] @ axA, X[sel] @ axB
res["半分A"] = (sA - sA.mean()) / sA.std()
res["半分B"] = (sB - sB.mean()) / sB.std()

n = res.groupby("party").size()
pm = res.groupby("party")[["半分A", "半分B"]].mean()[n >= 30]
print(pm.sort_values("半分A").round(2))
print("2本の物差しでの、党の並びの相関:", round(pm.corr().iloc[0, 1], 3))

# (2) 維新の発言のうち、左寄り・右寄りと判定されたものを3件ずつ読む
print("\n===== 維新:左寄りと判定された3件 → 右寄りと判定された3件 =====")
ish = res[res["party"] == "維新"].sort_values("言葉_会議調整")
for idx in list(ish.index[:3]) + list(ish.index[-3:]):
    print(round(res.loc[idx, "言葉_会議調整"], 2), df.loc[idx, "nameOfMeeting"])
    print(df.loc[idx, "text"][:200], "\n")

        半分A   半分B
party            
社民    -0.90 -1.11
れいわ   -0.77 -0.08
共産    -0.16 -1.01
立憲    -0.13 -0.02
維新    -0.05 -0.08
その他    0.02 -0.82
国民     0.05  0.29
自民     0.15  0.40
みらい    0.24  0.63
参政     0.30  0.46
中道     0.35  0.17
公明     0.45  0.56
2本の物差しでの、党の並びの相関: 0.66

===== 維新:左寄りと判定された3件 → 右寄りと判定された3件 =====
-2.84 外交防衛委員会
尖閣に日本人は上陸できない、日本の行政施設、何も造れない。こうなったら日本の施政権が及んでいないということになって、アメリカは、これ安保条約第五条の施政権下にあるというところに当てはまらないねと、竹島と余り変わらなくなっちゃっているねといったら、アメリカは助けてくれませんよ。 

-2.71 安全保障委員会
今の答弁は私は全く納得できません。 要するに、契約の時点で、遅れたらどうするとかペナルティーをかけるとかいうのは契約として当たり前のことじゃないですか。その当たり前のことを全くしていないというのはどういうことですか。大臣、どうですか。 

-2.55 外交防衛委員会
大臣の言わんとしていることは理解はできますけどね。 それでは、ちょっと遡って、トランプの一期目のときの国家安全保障担当の大統領補佐官を務めたボルトン氏の証言によると、当時、このトランプ大統領は日本側の安倍政権に従来の三倍超の年間八十億ドルもの在日米軍駐留経費を要求したという報道が、記事が出ていますけれども、これは事実ですか。 

2.69 外交防衛委員会
大変重要な組織再編だと思います。 最後に、時間があるのでもう一問聞きますが、陸上自衛隊の、今度、補給統制本部が補給本部に改編されて、各方面に所在する陸上自衛隊の補給処を指揮監督できる体制へ移行することとされています。 他方、陸上自衛隊の各方面隊には補給を含む後方支援を担う部隊が置かれると理解しています。これらの部隊と補給処の今後の連携の在り方についてどのように考えているか、お伺いしたいと思います。 

**結果**

- (1) 2本の物差しでの党の並びの相関は **0.66**。れいわ(−0.77 と −0.08)、共産(−0.16 と −1.01)、その他(0.02 と −0.82)のように、種の文を半分入れ替えただけで大きく動く → **不安定**
- (2) 「左寄り」と判定された維新の発言は、政府の答弁を批判したり問いただしたりする口調(「全く納得できません」など)。「右寄り」と判定された発言は、自衛隊の組織改編を確かめたり、「しっかり頑張っていただきたい」と協力的に締めくくったりする口調。どちらも安全保障を強めたい立場は同じ → **物差しは立場ではなく、話し方(批判的か協力的か)と話題を拾っている**

## 結論

- 争点をまたいだ共通の「右−左」の向きはない(−0.03)。BERT は否定の形を強い1つの向きとして捉える(±0.8)
- 言葉づかいをそろえた種の文では、種の文どうしは分けられる(19/20)。しかし実際の発言6,172件に当てると、種の文を半分入れ替えただけで党の並びの相関が 0.66 に下がり、不安定
- 物差しが拾っているのは、主に話し方(批判的か協力的か)と話題。公明の離脱前後の変化も、与党・野党の立場の変化と区別できない
- → **mean pooling の BERT 埋め込み + 種の文の方向では、立場を安定して取り出せない**。会議録 × BERT は党の位置の推定に使わない。党の位置は、マニフェストを政策ごとに分けて LLM で立場を判定する方法を主にし、マニフェスト × BERT は比較として試す(README に書く)

**限界**: 種の文は自分で書いたもので、書き方を変えれば結果も変わりうる(それ自体がこの方法の弱さ)。安全保障の委員会だけで確かめた